# Этап 3: Отслеживание Сноубордиста и Имитация Камеры Дрона

Этот ноутбук демонстрирует реализацию алгоритма отслеживания целевого объекта (сноубордиста) на видео с помощью модели YOLO11 и имитацию удержания объекта в центре кадра, симулируя работу камеры беспилотного летательного аппарата.

**Цель:** Создать выходное видео, где отслеживаемый сноубордист остается центрированным в кадре, показывая принцип работы автоматического слежения.

### 1. Подготовка Окружения и Импорты
Инициализация необходимых библиотек и функций из модулей проекта.

In [ ]:
# --- Импорты ---
import os
import sys
from IPython.display import Video

# Добавление корневой директории проекта в sys.path, позволяет импортировать модули из папок, расположенных на одном уровне с 'notebooks/', например из 'scripts/'
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# Импортируем функцию из скрипта для отслеживания и утилиту для генерации имен запусков
from scripts.tracker import track_video_and_center_object
from scripts.utils import get_next_run_name
from scripts.visualization_utils import create_side_by_side_demo_video

### 2. Конфигурация Пайплайна Отслеживания
Определение путей к данным, обученной модели и ключевых параметров для процесса отслеживания и вывода видео.

In [ ]:
# Путь к исходному видео
video_input_path = '../resources/snowboard_short.mp4'

# Путь к папке с обученной моделью YOLO11
model_folder_name = 'yolo11n_snowboarder_detection_v1'

model_path = os.path.join(project_root, 'runs', 'detect', model_folder_name, 'weights', 'best.pt')

# Определение имени для текущего запуска отслеживания
track_run_name = get_next_run_name("snowboarder_tracking", runs_relative_path=os.path.join(project_root, 'runs', 'track'))

# Путь для сохранения выходного видео с отслеживанием
video_output_path = os.path.join(project_root, 'runs', 'track', track_run_name, 'tracked_snowboarder.mp4')

# Параметры для отслеживания
target_class_id = 0           # ID класса 'snowboarder'
target_imgsz = 640            # Размер квадратного кадра для вывода
confidence_threshold = 0.25   # Порог уверенности для детекции
iou_threshold = 0.7           # Порог IoU для NMS

print("Настроены пути и параметры для текущего запуска:")
print(f"  Исходное видео: {video_input_path}")
print(f"  Модель для отслеживания: {model_path}")
print(f"  Выходное видео будет сохранено в: {video_output_path}")
print(f"  Имя текущего запуска отслеживания: {track_run_name}")

### 3. Выполнение Отслеживания и Центрирования
Запуск основной функции `track_video_and_center_object` для обработки видео.

In [ ]:
print("\nНачинаем процесс отслеживания и центрирования объекта...")

track_video_and_center_object(
    model_path=model_path,
    video_input_path=video_input_path,
    video_output_path=video_output_path,
    target_class_id=target_class_id,
    target_imgsz=target_imgsz,
    confidence_threshold=confidence_threshold,
    iou_threshold=iou_threshold
)

print("Процесс отслеживания завершен.")
print(f"Результаты можно найти в папке: {os.path.dirname(video_output_path)}/")

### 4. Создание демо-видео

In [ ]:
demo_output_path = os.path.join(os.path.dirname(video_output_path), f"demo_final_{track_run_name}.mp4")
tracking_log_path = os.path.join(os.path.dirname(video_output_path), "tracking_log.jsonl")

print(f"\nНачинаем создание демонстрационного видео: {demo_output_path}")
create_side_by_side_demo_video(
    original_video_path=video_input_path, # Путь к исходному видео
    tracked_video_path=video_output_path, # Путь к обработанному видео
    tracking_log_path=tracking_log_path,  # Путь к файлу логов
    output_demo_path=demo_output_path,
    tracked_video_fixed_size=640, # Размер квадратного видео
    overlay_height=120 # Высота оверлея
)

print("Создание демонстрационного видео завершено.")